In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ziyadaltalhi/google-maps-data/part-00010-ec7c49a0-d4f4-4864-bc16-3beaf6f5f6f1-c000.snappy.parquet
/kaggle/input/datasets/ziyadaltalhi/google-maps-data/.part-00005-ec7c49a0-d4f4-4864-bc16-3beaf6f5f6f1-c000.snappy.parquet.crc
/kaggle/input/datasets/ziyadaltalhi/google-maps-data/.part-00010-ec7c49a0-d4f4-4864-bc16-3beaf6f5f6f1-c000.snappy.parquet.crc
/kaggle/input/datasets/ziyadaltalhi/google-maps-data/part-00008-ec7c49a0-d4f4-4864-bc16-3beaf6f5f6f1-c000.snappy.parquet
/kaggle/input/datasets/ziyadaltalhi/google-maps-data/.part-00016-ec7c49a0-d4f4-4864-bc16-3beaf6f5f6f1-c000.snappy.parquet.crc
/kaggle/input/datasets/ziyadaltalhi/google-maps-data/part-00017-ec7c49a0-d4f4-4864-bc16-3beaf6f5f6f1-c000.snappy.parquet
/kaggle/input/datasets/ziyadaltalhi/google-maps-data/.part-00015-ec7c49a0-d4f4-4864-bc16-3beaf6f5f6f1-c000.snappy.parquet.crc
/kaggle/input/datasets/ziyadaltalhi/google-maps-data/.part-00000-ec7c49a0-d4f4-4864-bc16-3beaf6f5f6f1-c000.snappy.parquet.crc
/kaggle

In [2]:
# install libraries
!pip install -q transformers datasets evaluate accelerate pyarrow

# imports
import os
import inspect
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import classification_report, confusion_matrix

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

# check device
USE_CUDA = torch.cuda.is_available()

print("CUDA available:", USE_CUDA)
print("GPU count:", torch.cuda.device_count())

if USE_CUDA:
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not detected by PyTorch. The code will continue on CPU.")

# path
parquet_path = "/kaggle/input/datasets/ziyadaltalhi/google-maps-data"

if not os.path.exists(parquet_path):
    raise FileNotFoundError(f"Path not found: {parquet_path}")

# read parquet
df = pd.read_parquet(parquet_path)

print("Original shape:", df.shape)
print(df.columns.tolist())

# select text column
TEXT_CANDIDATES = ["Text_TR", "Text7", "Text_Orig", "text23", "Text", "text"]

text_col = None
for c in TEXT_CANDIDATES:
    if c in df.columns:
        text_col = c
        break

if text_col is None:
    raise ValueError("Text column not found")

print("Using text column:", text_col)

df["text"] = df[text_col].astype(str)
df = df[df["text"].notna()]
df = df[df["text"].str.strip() != ""]

# labels
if "Sentiment" in df.columns:
    df["Sentiment"] = df["Sentiment"].astype(str).str.lower().str.strip()
    df = df[df["Sentiment"].isin(["negative", "positive"])]
    df["label"] = df["Sentiment"].map({"negative": 0, "positive": 1})

elif "Stars" in df.columns or "stars" in df.columns:
    star_col = "Stars" if "Stars" in df.columns else "stars"
    df[star_col] = pd.to_numeric(df[star_col], errors="coerce")
    df = df[df[star_col].notna()]
    df = df[df[star_col] != 3]
    df["label"] = np.where(df[star_col] <= 2, 0, 1)

else:
    raise ValueError("No label column found")

df = df[["text", "label"]].dropna()
df["label"] = df["label"].astype(int)

print("Final shape:", df.shape)
print(df["label"].value_counts())

# optional sample
SAMPLE_SIZE = None
# SAMPLE_SIZE = 100000

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42)

# split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# tokenizer
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-da"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

# model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# training args
args_dict = dict(
    output_dir="/kaggle/working/camelbert_results",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none",
    fp16=USE_CUDA,
    eval_accumulation_steps=10
)

signature = inspect.signature(TrainingArguments.__init__)

if "eval_strategy" in signature.parameters:
    args_dict["eval_strategy"] = "epoch"
else:
    args_dict["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**args_dict)

# trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# train
trainer.train()

# evaluate
results = trainer.evaluate()
print(results)

predictions_output = trainer.predict(test_dataset)

y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = predictions_output.label_ids

print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"],
    zero_division=0
))

print(confusion_matrix(y_true, y_pred))

# save
SAVE_PATH = "/kaggle/working/camelbert_model"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved:", SAVE_PATH)

# test
def predict_sentiment(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    return "positive" if pred == 1 else "negative"

print(predict_sentiment("The place is beautiful and service is excellent"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
CUDA available: True
GPU count: 2
GPU name: Tesla T4
Original shape: (1016595, 81)
['CategoryName0', 'Location/lat1', 'Location/lng2', 'Neighborhood3', 'Date', 'Stars', 'City6', 'Text7', 'Place Name', 'Street9', 'Region', 'Text_Orig', 'Emoji_List', 'Emoji_Count', 'Emoji_Pos_Count', 'Emoji_Neg_Count', 'Emoji_Score', 'Emoji_Sentiment', 'Text_Base', 'Text_TR', 'Text_ML', 'Is_Short_TR', 'Is_Short_ML', 'text23', 'categoryName24', 'city25', 'location/lat26', 'location/lng27', 'neighborhood28', 'publishedAtDate', 'title', 'street31', 'address', 'cid', 'fid', 'imageUrl', 'isAdvertisement', 'isLocalGuide', 'language', 'likesCount', 'originalLanguage', 'permanentlyClosed', 'placeId', 'postalCode', 'price', 'publishAt', 'rating', 'responseFromOwnerDate', 'responseFromOwnerText', 'reviewId', 'reviewImageUrls/0', 'reviewImageUrls/1', 'reviewImageUrls/2', 'reviewImageUrls/3', 'reviewImageUrls/4', 'reviewImageUrls/5', 'revi

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

{'eval_loss': 0.49214425683021545, 'eval_accuracy': 0.9282069252502245, 'eval_precision': 0.9221627777947026, 'eval_recall': 0.9282069252502245, 'eval_f1': 0.9190167121028391, 'eval_runtime': 936.1389, 'eval_samples_per_second': 195.095, 'eval_steps_per_second': 6.097, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

    negative       0.81      0.45      0.58     20121
    positive       0.94      0.99      0.96    162515

    accuracy                           0.93    182636
   macro avg       0.87      0.72      0.77    182636
weighted avg       0.92      0.93      0.92    182636

[[  9132  10989]
 [  2123 160392]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved: /kaggle/working/camelbert_model
positive


In [3]:
import shutil

shutil.make_archive("/kaggle/working/camelbert_model", 'zip', "/kaggle/working/camelbert_model")

'/kaggle/working/camelbert_model.zip'

In [1]:
# install libraries
!pip install -q transformers accelerate pyarrow

# imports
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from IPython.display import display
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# paths
parquet_path = "/kaggle/input/datasets/ziyadaltalhi/google-maps-data"
output_dir = "/kaggle/working"

if not os.path.exists(parquet_path):
    raise FileNotFoundError(f"Path not found: {parquet_path}")

# read parquet
df = pd.read_parquet(parquet_path)

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

# select text column
TEXT_CANDIDATES = ["Text_TR", "Text7", "Text_Orig", "text23", "Text", "text"]

text_col = None
for c in TEXT_CANDIDATES:
    if c in df.columns:
        text_col = c
        break

if text_col is None:
    raise ValueError("Text column not found")

print("Using text column:", text_col)

# normalize metadata columns
def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

category_col = first_existing_column(df, ["CategoryName", "CategoryName0", "categoryName24", "categoryName"])
lat_col = first_existing_column(df, ["Location/lat", "Location/lat1", "location/lat26", "location/lat", "lat"])
lng_col = first_existing_column(df, ["Location/lng", "Location/lng2", "location/lng27", "location/lng", "lng"])
city_col = first_existing_column(df, ["City", "City6", "city25", "city"])
region_col = first_existing_column(df, ["Region", "Region_Name"])
date_col = first_existing_column(df, ["Date", "publishedAtDate", "publishAt"])
stars_col = first_existing_column(df, ["Stars", "stars", "Stars_clean", "rating"])

if category_col is not None:
    df["CategoryName"] = df[category_col]

if lat_col is not None:
    df["Lat"] = df[lat_col]

if lng_col is not None:
    df["Lng"] = df[lng_col]

if city_col is not None:
    df["City_Folder"] = df[city_col]

if region_col is not None:
    df["Region_Name"] = df[region_col]

if date_col is not None:
    df["Date"] = df[date_col]

if stars_col is not None:
    df["Stars"] = df[stars_col]

# filter empty text
empty_like = {
    "", "nan", "NaN", "none", "None", "NONE",
    "<NA>", "[]", "[ ]", "{}", "null", "NULL"
}

s = df[text_col]

empty_mask = (
    s.isna() |
    s.astype(str).str.strip().isin(empty_like)
)

df2 = df.loc[~empty_mask].copy()
df2["TEXT_FOR_MODEL"] = df2[text_col].astype(str).str.strip()

print("Rows before text filter:", len(df))
print("Rows after text filter:", len(df2))

# safe Arabic cleaning
AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
AR_TATWEEL_RE = re.compile(r"\u0640")

def normalize_arabic_safe(text):
    text = AR_DIACRITICS_RE.sub("", text)
    text = AR_TATWEEL_RE.sub("", text)
    text = re.sub(r"[إأآا]", "ا", text)
    return text

def clean_text(s):
    if pd.isna(s):
        return ""

    s = str(s).strip().lower()

    if not s:
        return ""

    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"\S+@\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"#\w+", " ", s)
    s = re.sub(r"[\U00010000-\U0010ffff]", " ", s)
    s = re.sub(r"[^0-9a-z\u0600-\u06FF\s]", " ", s)

    s = normalize_arabic_safe(s)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    s = re.sub(r"\s+", " ", s).strip()

    if re.fullmatch(r"\d+", s):
        return ""

    return s

# keep metadata
META_COLS = [
    "Stars", "Region_Name", "City_Folder",
    "CategoryName", "Lat", "Lng", "Date"
]

meta_existing = [c for c in META_COLS if c in df2.columns]

df4 = df2[[text_col] + meta_existing].copy()
df4 = df4.rename(columns={text_col: "text_before"})
df4["text_clean"] = df4["text_before"].apply(clean_text)

df_removed = df4[df4["text_clean"].str.len() == 0].copy()
df4 = df4[df4["text_clean"].str.len() > 0].copy()

print("Removed rows after cleaning:", len(df_removed))
print("Final cleaned shape:", df4.shape)

display(df4.head())

# aspect dictionary
ASPECT_PATTERNS = {
    "النظافة": [
        r"(نظافه|نظافة|نظيف|نظيفه|نظيفة|متسخ|متسخه|وسخ|وسخه|وصخ|وصخه|قذر|قذره|قذارة|زباله|قمامه|نفايات|تعقيم|معقم|روائح|ريحه\s*كريهه|حشرات|ذباب|صراصير)"
    ],
    "دورات المياه": [
        r"(دورات\s*المياه|دوره\s*مياه|دورة\s*مياه|حمام|الحمام|حمامات|تو[اا]ليت|مرحاض|مغسله|مغاسل|صابون|مناديل|ورق\s*تواليت)"
    ],
    "الخدمة": [
        r"(خدمه|خدمة|الخدمه|الخدمة|تعامل|التعامل|اسلوب|أسلوب|استقبال|موظف|موظفين|العامل|العمال|الكاشير|الاداره|الإدارة|متعاون|محترم|وقح|تجاهل)"
    ],
    "مواقف السيارات": [
        r"(مواقف|موقف|مواقف\s*السيارات|باركنج|مافي\s*مواقف|ما\s*في\s*مواقف|مواقف\s*قليله|مواقف\s*محدوده)"
    ],
    "الانتظار": [
        r"(انتظار|الانتظار|وقت\s*انتظار|طابور|صف|الدور|تاخير|تأخير|يتاخر|يتأخر|بطيء|بطيئ|بطئ|سريع|سرعه)"
    ],
    "الأسعار": [
        r"(سعر|اسعار|أسعار|الاسعار|الأسعار|غالي|غاليه|غالية|رخيص|رخيصه|رخيصة|مناسب|مناسبه|مبالغ\s*فيه|استغلال|خصم|عروض)"
    ],
    "الطعام": [
        r"(اكل|أكل|الاكل|الأكل|طعام|الطعام|وجبه|وجبة|وجبات|طعم|طعمه|مذاق|لذيذ|لذيذه|طازج|بارد|حار|محروق|ناشف|مالح|صوص|بهارات)"
    ],
    "الإضاءة": [
        r"(اضاءه|إضاءة|الاضاءه|الإضاءة|اناره|إنارة|نور|الانوار|الأنوار|لمبه|لمبات|مظلم|ظلام|معتم)"
    ],
    "الزحمة": [
        r"(زحمه|زحمة|زحام|ازدحام|مزدحم|مكتظ|كتمه|خانقه|ضيق|مساحه\s*ضيقه|جلسات\s*قليله)"
    ],
    "الألعاب": [
        r"(العاب|ألعاب|منطقه\s*العاب|منطقة\s*ألعاب|ملاهي|ترفيه|اركيد|أركيد|بلايستيشن|بلياردو|بولينج|العاب\s*اطفال)"
    ],
    "الصيانة": [
        r"(صيانه|صيانة|الصيانه|الصيانة|عطل|اعطال|أعطال|خربان|خربانه|مو\s*شغال|لا\s*يعمل|تصليح|اصلاح|إصلاح|تسريب)"
    ]
}

ASPECT_REGEX = {
    asp: [re.compile(pat) for pat in pats]
    for asp, pats in ASPECT_PATTERNS.items()
}

def detect_aspects_with_hits(text):
    if not isinstance(text, str):
        return {}

    found = {}

    for asp, regs in ASPECT_REGEX.items():
        hits = []

        for rg in regs:
            for m in rg.finditer(text):
                hits.append(m.group(0))

        if hits:
            found[asp] = sorted(set(hits))

    return found

df4["aspect_hits"] = df4["text_clean"].apply(detect_aspects_with_hits)

# create aspect long dataframe
rows = []

for idx, r in df4.iterrows():
    hits_dict = r["aspect_hits"]

    if not hits_dict:
        continue

    for aspect, hits in hits_dict.items():
        row = {
            "row_id": idx,
            "aspect": aspect,
            "hits": " | ".join(hits),
            "text": r["text_clean"]
        }

        for c in meta_existing:
            row[c] = r[c]

        rows.append(row)

df_aspect_long = pd.DataFrame(rows)

print("Aspect-level rows:", len(df_aspect_long))
display(df_aspect_long.head(20))

print("Aspect distribution:")
display(df_aspect_long["aspect"].value_counts())

# aspect analytics
aspect_overall = (
    df_aspect_long
    .groupby("aspect")
    .size()
    .reset_index(name="mentions")
    .sort_values("mentions", ascending=False)
)

total_mentions = aspect_overall["mentions"].sum()
aspect_overall["percentage_%"] = (aspect_overall["mentions"] / total_mentions * 100).round(2)

display(aspect_overall)

aspect_unique_reviews = (
    df_aspect_long
    .groupby("aspect")["row_id"]
    .nunique()
    .reset_index(name="unique_reviews")
    .sort_values("unique_reviews", ascending=False)
)

display(aspect_unique_reviews)

if "Region_Name" in df_aspect_long.columns:
    aspect_by_region_mentions = (
        df_aspect_long
        .groupby(["Region_Name", "aspect"])
        .size()
        .reset_index(name="mentions")
        .sort_values(["Region_Name", "mentions"], ascending=[True, False])
    )

    display(aspect_by_region_mentions.head(30))

if "City_Folder" in df_aspect_long.columns:
    aspect_by_city_mentions = (
        df_aspect_long
        .groupby(["City_Folder", "aspect"])
        .size()
        .reset_index(name="mentions")
    )

    top_aspects_city = (
        aspect_by_city_mentions
        .sort_values(["City_Folder", "mentions"], ascending=[True, False])
        .groupby("City_Folder")
        .head(3)
    )

    display(top_aspects_city)

# sentiment model
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
MAX_LEN = 128
BATCH_SIZE = 64
SAT_MODE = "pos_only"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("id2label:", model.config.id2label)

id2label = {int(k): v for k, v in model.config.id2label.items()}
label_lower = {i: str(lab).lower() for i, lab in id2label.items()}

def find_idx(keys):
    for i, lab in label_lower.items():
        if any(k in lab for k in keys):
            return i
    return None

pos_idx = find_idx(["pos", "positive", "ايجاب", "إيجاب"])
neu_idx = find_idx(["neu", "neutral", "محايد"])
neg_idx = find_idx(["neg", "negative", "سلب", "سلبي"])

if pos_idx is None or neu_idx is None or neg_idx is None:
    pos_idx, neu_idx, neg_idx = 2, 1, 0

print("Indices -> pos:", pos_idx, "neu:", neu_idx, "neg:", neg_idx)

# aspect-aware input
df_aspect_long["bert_input"] = (
    df_aspect_long["aspect"].astype(str) + " [SEP] " + df_aspect_long["text"].astype(str)
)

df_aspect_long = df_aspect_long[df_aspect_long["bert_input"].str.len() > 0].copy()
df_aspect_long.reset_index(drop=True, inplace=True)

print("After bert_input:", df_aspect_long.shape)

# batch inference
def batch_predict_3class(texts, batch_size=64, max_len=128, log_every_batches=300):
    all_pos, all_neu, all_neg = [], [], []

    N = len(texts)

    for b, start in enumerate(range(0, N, batch_size), 1):
        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits

        probs = F.softmax(logits, dim=1)
        probs = probs.detach().cpu().numpy()

        all_pos.append(probs[:, pos_idx])
        all_neu.append(probs[:, neu_idx])
        all_neg.append(probs[:, neg_idx])

        if b % log_every_batches == 0:
            done = min(start + batch_size, N)
            print(f"Processed {done}/{N} rows")

    p_pos = np.concatenate(all_pos, axis=0)
    p_neu = np.concatenate(all_neu, axis=0)
    p_neg = np.concatenate(all_neg, axis=0)

    return p_pos, p_neu, p_neg

texts = df_aspect_long["bert_input"].tolist()

p_pos, p_neu, p_neg = batch_predict_3class(
    texts,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
    log_every_batches=300
)

df_aspect_long["p_positive"] = p_pos
df_aspect_long["p_neutral"] = p_neu
df_aspect_long["p_negative"] = p_neg

pred_idx = np.argmax(
    np.stack([
        df_aspect_long["p_negative"],
        df_aspect_long["p_neutral"],
        df_aspect_long["p_positive"]
    ], axis=1),
    axis=1
)

df_aspect_long["sentiment"] = np.where(
    pred_idx == 2,
    "positive",
    np.where(pred_idx == 1, "neutral", "negative")
)

if SAT_MODE == "pos_only":
    df_aspect_long["satisfaction_score"] = df_aspect_long["p_positive"]
elif SAT_MODE == "pos_plus_half_neu":
    df_aspect_long["satisfaction_score"] = (
        df_aspect_long["p_positive"] + 0.5 * df_aspect_long["p_neutral"]
    )
else:
    raise ValueError("Invalid SAT_MODE")

print("Inference done")
display(df_aspect_long.head(10))

# satisfaction stats
aspect_satisfaction = (
    df_aspect_long
    .groupby("aspect")
    .agg(
        mentions=("aspect", "size"),
        unique_reviews=("row_id", "nunique"),
        avg_p_positive=("p_positive", "mean"),
        avg_p_neutral=("p_neutral", "mean"),
        avg_p_negative=("p_negative", "mean"),
        satisfaction=("satisfaction_score", "mean")
    )
    .reset_index()
)

aspect_satisfaction["satisfaction_%"] = (aspect_satisfaction["satisfaction"] * 100).round(2)
aspect_satisfaction = aspect_satisfaction.sort_values("satisfaction_%", ascending=True)

print("Overall Aspect Satisfaction")
display(aspect_satisfaction)

aspect_by_region = None
if "Region_Name" in df_aspect_long.columns:
    aspect_by_region = (
        df_aspect_long
        .groupby(["Region_Name", "aspect"])
        .agg(
            mentions=("aspect", "size"),
            unique_reviews=("row_id", "nunique"),
            satisfaction=("satisfaction_score", "mean"),
            avg_p_positive=("p_positive", "mean"),
            avg_p_negative=("p_negative", "mean")
        )
        .reset_index()
    )

    aspect_by_region["satisfaction_%"] = (aspect_by_region["satisfaction"] * 100).round(2)

    print("Aspect Satisfaction by Region")
    display(aspect_by_region.head(30))

aspect_by_city = None
if "City_Folder" in df_aspect_long.columns:
    aspect_by_city = (
        df_aspect_long
        .groupby(["City_Folder", "aspect"])
        .agg(
            mentions=("aspect", "size"),
            unique_reviews=("row_id", "nunique"),
            satisfaction=("satisfaction_score", "mean"),
            avg_p_positive=("p_positive", "mean"),
            avg_p_negative=("p_negative", "mean")
        )
        .reset_index()
    )

    aspect_by_city["satisfaction_%"] = (aspect_by_city["satisfaction"] * 100).round(2)

    print("Aspect Satisfaction by City")
    display(aspect_by_city.head(30))

# save outputs
df_aspect_long.drop(columns=["bert_input"], errors="ignore").to_csv(
    f"{output_dir}/aspect_sentiment_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

aspect_overall.to_csv(
    f"{output_dir}/aspect_overall_mentions.csv",
    index=False,
    encoding="utf-8-sig"
)

aspect_unique_reviews.to_csv(
    f"{output_dir}/aspect_unique_reviews.csv",
    index=False,
    encoding="utf-8-sig"
)

aspect_satisfaction.to_csv(
    f"{output_dir}/aspect_satisfaction.csv",
    index=False,
    encoding="utf-8-sig"
)

if aspect_by_region is not None:
    aspect_by_region.to_csv(
        f"{output_dir}/aspect_satisfaction_by_region.csv",
        index=False,
        encoding="utf-8-sig"
    )

if aspect_by_city is not None:
    aspect_by_city.to_csv(
        f"{output_dir}/aspect_satisfaction_by_city.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("Saved to /kaggle/working")

Original shape: (1016595, 81)
Columns: ['CategoryName0', 'Location/lat1', 'Location/lng2', 'Neighborhood3', 'Date', 'Stars', 'City6', 'Text7', 'Place Name', 'Street9', 'Region', 'Text_Orig', 'Emoji_List', 'Emoji_Count', 'Emoji_Pos_Count', 'Emoji_Neg_Count', 'Emoji_Score', 'Emoji_Sentiment', 'Text_Base', 'Text_TR', 'Text_ML', 'Is_Short_TR', 'Is_Short_ML', 'text23', 'categoryName24', 'city25', 'location/lat26', 'location/lng27', 'neighborhood28', 'publishedAtDate', 'title', 'street31', 'address', 'cid', 'fid', 'imageUrl', 'isAdvertisement', 'isLocalGuide', 'language', 'likesCount', 'originalLanguage', 'permanentlyClosed', 'placeId', 'postalCode', 'price', 'publishAt', 'rating', 'responseFromOwnerDate', 'responseFromOwnerText', 'reviewId', 'reviewImageUrls/0', 'reviewImageUrls/1', 'reviewImageUrls/2', 'reviewImageUrls/3', 'reviewImageUrls/4', 'reviewImageUrls/5', 'reviewImageUrls/6', 'reviewImageUrls/7', 'reviewImageUrls/8', 'reviewImageUrls/9', 'reviewImageUrls/10', 'reviewImageUrls/11',

,text_before,Stars,Region_Name,City_Folder,CategoryName,Lat,Lng,Date,text_clean
0,تم تطويره عن السابق و الشيء المميز لا يوجد قرو...,5,None,None,None,None,None,None,تم تطويره عن السابق و الشيء المميز لا يوجد قرو...
2,الاول,5,None,None,None,None,None,None,الاول
4,ممتاز,5,None,None,None,None,None,None,ممتاز
5,موقع في غاية الروعة يستحق العناء والزيارة,5,None,None,None,None,None,None,موقع في غاية الروعة يستحق العناء والزيارة
9,جميل المكان والجو,5,None,None,None,None,None,None,جميل المكان والجو


Aspect-level rows: 260184


,row_id,aspect,hits,text,Stars,Region_Name,City_Folder,CategoryName,Lat,Lng,Date
0,19,الطعام,بارد,المكان جميل جدا وابارد,5,None,None,None,None,None,None
1,29,الألعاب,العاب,المكان الوحيد اللي اروح له و انا متطمنه ان بنت...,5,None,None,None,None,None,None
2,37,الألعاب,العاب,مكان جميل جدا، عبارة عن غابة مسوين فيها جلسات ...,5,None,None,None,None,None,None
3,40,النظافة,نظافة,مكان عائلي جميل بس ياليت تهتمو بنظافة دورات ال...,4,None,None,None,None,None,None
4,40,دورات المياه,دورات المياه,مكان عائلي جميل بس ياليت تهتمو بنظافة دورات ال...,4,None,None,None,None,None,None
5,50,الأسعار,مناسب,منتزه جدا جميل ورايق لابعد حد، تبغي جلسات موجو...,5,None,None,None,None,None,None
6,53,الصيانة,عطل,منتزه جميل وخاصة بعد انتهاء العطله الصيفيه اصب...,5,None,None,None,None,None,None
7,58,الخدمة,محترم | موظف,يفوز ع جميع الاصعده وموظفينه محترمين,5,None,None,None,None,None,None
8,59,الطعام,اكل | طعم,المطعم للاسف اكل سيء والشاورما مستويه زيادة عا...,1,None,None,None,None,None,None
9,69,الصيانة,عطل,منتزه رغدان مكان هادئ ومريح، مثالي لمن يرغبون ...,5,None,None,None,None,None,None


Aspect distribution:


...

Aspect Satisfaction by Region


,Region_Name,aspect,mentions,unique_reviews,satisfaction,avg_p_positive,avg_p_negative,satisfaction_%
0,Al Jouf,الأسعار,268,268,0.544668,0.544668,0.289637,54.470001
1,Al Jouf,الألعاب,13,13,0.627011,0.627011,0.172386,62.700001
2,Al Jouf,الإضاءة,11,11,0.620127,0.620127,0.085789,62.009998
3,Al Jouf,الانتظار,96,96,0.457391,0.457391,0.370008,45.740002
4,Al Jouf,الخدمة,286,286,0.584613,0.584613,0.298374,58.459999
5,Al Jouf,الزحمة,37,37,0.287636,0.287636,0.526583,28.760000
6,Al Jouf,الصيانة,35,35,0.422510,0.422510,0.324522,42.250000
7,Al Jouf,الطعام,432,432,0.649818,0.649818,0.191283,64.980003
8,Al Jouf,النظافة,95,95,0.644279,0.644279,0.245021,64.430000
9,Al Jouf,دورات المياه,10,10,0.507844,0.507844,0.261833,50.779999


Aspect Satisfaction by City


,City_Folder,aspect,mentions,unique_reviews,satisfaction,avg_p_positive,avg_p_negative,satisfaction_%
0,Al Ju'ranah,الأسعار,1,1,0.094143,0.094143,0.069788,9.410000
1,Al Ju'ranah,الألعاب,10,10,0.365204,0.365204,0.229973,36.520000
2,Al Ju'ranah,الإضاءة,7,7,0.314673,0.314673,0.517221,31.469999
3,Al Ju'ranah,الانتظار,2,2,0.418217,0.418217,0.497794,41.820000
4,Al Ju'ranah,الخدمة,1,1,0.820927,0.820927,0.021306,82.089996
5,Al Ju'ranah,الزحمة,1,1,0.797202,0.797202,0.057554,79.720001
6,Al Ju'ranah,الصيانة,2,2,0.478128,0.478128,0.497877,47.810001
7,Al Ju'ranah,الطعام,2,2,0.269534,0.269534,0.052404,26.950001
8,Al Ju'ranah,النظافة,10,10,0.739187,0.739187,0.070988,73.919998
9,Al Ju'ranah,دورات المياه,15,15,0.447896,0.447896,0.180688,44.790001


Saved to /kaggle/working
